In [18]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:openai/gpt-oss-20b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.2', 'langchain': '1.4.0'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x00000238A29D7750>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000238A29D7ED0>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [6]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title:str=Field(description="This is the title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="This is the name of director of movie")
    rating:float=Field(description="The movie's rating out of 10")

In [7]:
model_with_structured=model.with_structured_output(Movie)
model_with_structured

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.2', 'langchain': '1.4.0'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001B127E37B60>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001B12806C6E0>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'This is 

In [9]:
model_with_structured.invoke("Provide details about the movie 1920")

Movie(title='1920', year=2008, director='Abhishek Sharma', rating=6.5)

### Message output alongside parsed structure

In [14]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title:str=Field(description="The Title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="This is the name of director of movie")
    rating:float=Field(description="The movie's rating out of 10")

model_with_structure=model.with_structured_output(Movie,include_raw=True)


response=model_with_structure.invoke("Provide me the details of the movie Iron Man")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to call function.', 'tool_calls': [{'id': 'fc_cd54158d-01b5-44a5-9379-6c104069a7ea', 'function': {'arguments': '{"director":"Jon Favreau","rating":8.4,"title":"Iron Man","year":2008}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 169, 'total_tokens': 216, 'completion_time': 0.061577952, 'completion_tokens_details': {'reasoning_tokens': 7}, 'prompt_time': 0.012752844, 'prompt_tokens_details': None, 'queue_time': 0.307827813, 'total_time': 0.074330796}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_4f7e7dc26e', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a08e5d-189e-7030-9c6b-a43f163c1ceb-0', tool_calls=[{'name': 'Movie', 'args': {'director': 'Jon Favreau', 'rating': 8.4, 'title': 'Iron Man', 'year': 2008}, 'id': 'fc_cd54158d-01b5-44a5-9379-6c104069

### Nested Structure


In [3]:
from pydantic import  BaseModel, Field
class Actor(BaseModel):
    name:str
    role:str
class MovieDetails(BaseModel):
    title: str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float | None=Field(None,description="Budget in million USD")


model_with_structure=model.with_structured_output(MovieDetails)
# model_with_structure=model.with_structured_output(Movie)
response=model_with_structure.invoke("Provide details about the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Cillian Murphy', role='Robert Fischer')], genres=['Action', 'Adventure', 'Science Fiction', 'Thriller'], budget=160000000.0)

### TypeDict

In [7]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details"""
    title: Annotated[str,...,"The title of the movie"]
    year:Annotated[int,...,"The year the movie was released"]
    director:Annotated[str,...,"The director of the movie"]
    rating:Annotated[float,...,"The movie's rating out of 10"]

model_typeDict_structure=model.with_structured_output(MovieDict)

response=model_typeDict_structure.invoke("Provide me the details of the movie Avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'Avengers', 'year': 2012}

In [8]:

class Actor(TypedDict):
    name:str
    role:str
class MovieDetails(TypedDict):
    title: str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float | None=Field(None,description="Budget in million USD")


model_with_structure=model.with_structured_output(MovieDetails)
# model_with_structure=model.with_structured_output(Movie)
response=model_with_structure.invoke("Provide details about the movie Avengers age of ultron")
response

{'budget': 365000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'},
  {'name': 'Tom Hiddleston', 'role': 'Loki'},
  {'name': 'Samuel L. Jackson', 'role': 'Nick Fury'},
  {'name': 'Paul Rudd', 'role': 'Scott Lang / Ant‑Man'},
  {'name': 'Zoe Saldana', 'role': 'Gamora'},
  {'name': 'Michael B. Jordan', 'role': 'Vision'},
  {'name': 'Elizabeth Olsen', 'role': 'Wanda Maximoff / Scarlet Witch'}],
 'genres': ['Action', 'Adventure', 'Science Fiction', 'Fantasy'],
 'title': 'Avengers: Age of Ultron',
 'year': 2015}

In [10]:
model.profile

{'name': 'GPT OSS 20B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

### Data Classes

In [17]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
class ContactInfo(BaseModel):
    """Contact information of a person"""
    name:str=Field(description="The name of the person")
    email:str=Field(description="The email address of the person")
    phone:str=Field(description="The phone number of the person")

agent = create_agent(
    model=model,
    response_format=ContactInfo
)
result=agent.invoke({
    "messages":[{"role":"user","content":"Extract contact from : John Doe, john@example.com,(555)123-4567"}]
})


result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555)123-4567')

In [21]:
## TypeDict

from typing_extensions import TypedDict, Annotated
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """Contact information of a person"""
    name:str=  str  # "The name of the person"
    email:str= str  #"The email address of the person"
    phone:str= str  #"The phone number of the person"

agent=create_agent(
    model=model,
    response_format=ContactInfo # Auto-selects ProviderStrategy
)

result=agent.invoke({
    "messages":[{"role":"user","content":"Extract contact from : John Doe, john@example.com,(555)123-4567"}]
})

result["structured_response"]


{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555)123-4567'}

In [23]:
#Data Class


from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person"""
    name:str 
    email:str
    phone:str

agent=create_agent(
    model=model,
    response_format=ContactInfo
)

result=agent.invoke({
    'messages':[{"role":"user","content":"Extract contact from : John Doe, john@example.com,(555)123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555)123-4567')